### Document Structure




In [64]:
from langchain_core.documents import Document

doc=Document(
    page_content="This is a test document.",
    metadata={
        "source": "test.txt",
        "author": "Ankit Mishra",
        "pages": 1
    }
)
print(doc)


page_content='This is a test document.' metadata={'source': 'test.txt', 'author': 'Ankit Mishra', 'pages': 1}


### Text Loader

In [65]:
from langchain_community.document_loaders import TextLoader

loader=TextLoader("../data/text_files/llm_intro.txt", encoding="utf-8")
document=loader.load()
document

[Document(metadata={'source': '../data/text_files/llm_intro.txt'}, page_content='What is an LLM?\nA Large Language Model (LLM) is an artificial intelligence program trained on massive amounts of text data [1]. It uses deep learning algorithms, specifically the transformer architecture, to understand, summarize, generate, and predict new content. Instead of understanding words like humans do, an LLM converts text into numerical values called tokens. It calculates the statistical probability of which token should follow next in a sequence based on its training. This allows it to write essays, write code, and answer questions, but its knowledge is frozen at the moment its training finishes.\nWhat is RAG?\nRetrieval-Augmented Generation (RAG) is a framework that optimizes the output of an LLM by fetching information from an external knowledge base before generating a response. When a user submits a query, a RAG system searches outside sources—such as company documents, databases, or live w

### Directory Loader

In [66]:
from langchain_community.document_loaders import DirectoryLoader

dir_loader=DirectoryLoader(
    "../data/text_files",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    show_progress=False
)
document=dir_loader.load()
document

[Document(metadata={'source': '..\\data\\text_files\\llm_intro.txt'}, page_content='What is an LLM?\nA Large Language Model (LLM) is an artificial intelligence program trained on massive amounts of text data [1]. It uses deep learning algorithms, specifically the transformer architecture, to understand, summarize, generate, and predict new content. Instead of understanding words like humans do, an LLM converts text into numerical values called tokens. It calculates the statistical probability of which token should follow next in a sequence based on its training. This allows it to write essays, write code, and answer questions, but its knowledge is frozen at the moment its training finishes.\nWhat is RAG?\nRetrieval-Augmented Generation (RAG) is a framework that optimizes the output of an LLM by fetching information from an external knowledge base before generating a response. When a user submits a query, a RAG system searches outside sources—such as company documents, databases, or liv

### PDF Loader

In [67]:
from langchain_community.document_loaders import PyMuPDFLoader

dir_loader=DirectoryLoader(
    "../data/pdf",
    glob="*.pdf",
    loader_cls=PyMuPDFLoader,
    show_progress=False 
)
pdf_documents=dir_loader.load()
pdf_documents

[Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-03-07T23:57:32+05:30', 'source': '..\\data\\pdf\\Aman_Resume-2026.pdf', 'file_path': '..\\data\\pdf\\Aman_Resume-2026.pdf', 'total_pages': 3, 'format': 'PDF 1.7', 'title': '', 'author': 'Apache POI', 'subject': '', 'keywords': '', 'moddate': '2026-03-07T23:57:32+05:30', 'trapped': '', 'modDate': "D:20260307235732+05'30'", 'creationDate': "D:20260307235732+05'30'", 'page': 0}, page_content='Aman Kumar Singh \n +91 7979010191 | \n amanhcv@gmail.com \nhttps://www.linkedin.com/in/amandotsingh/ \nSUMMARY \nExperienced Database Administrator with 4.6+ years of expertise in Oracle 19c, DBMS, and Exadata. \nSpecializes in performance tuning, backup/recovery, and high-availability systems. Proficient in tools \nlike Apica (External, Internal, Dynamic), GBT Dashboard, Grafana, ServiceNow, Java and Networking \nSupport, BigPanda, Jira and PagerDuty. \n \nWORK EXPERIENCE \nSenior Techno

### Chunking

In [68]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len
)

def chunk_documents(documents):
    chunked_documents = []
    for doc in documents:
        chunks = text_splitter.split_text(doc.page_content)
        for idx, chunk in enumerate(chunks):
            metadata = dict(doc.metadata)
            metadata["chunk_index"] = idx
            chunked_documents.append(Document(page_content=chunk, metadata=metadata))
    return chunked_documents

chunks = chunk_documents(pdf_documents)
chunks
    


[Document(metadata={'producer': 'Microsoft® Word 2021', 'creator': 'Microsoft® Word 2021', 'creationdate': '2026-03-07T23:57:32+05:30', 'source': '..\\data\\pdf\\Aman_Resume-2026.pdf', 'file_path': '..\\data\\pdf\\Aman_Resume-2026.pdf', 'total_pages': 3, 'format': 'PDF 1.7', 'title': '', 'author': 'Apache POI', 'subject': '', 'keywords': '', 'moddate': '2026-03-07T23:57:32+05:30', 'trapped': '', 'modDate': "D:20260307235732+05'30'", 'creationDate': "D:20260307235732+05'30'", 'page': 0, 'chunk_index': 0}, page_content='Aman Kumar Singh \n +91 7979010191 | \n amanhcv@gmail.com \nhttps://www.linkedin.com/in/amandotsingh/ \nSUMMARY \nExperienced Database Administrator with 4.6+ years of expertise in Oracle 19c, DBMS, and Exadata. \nSpecializes in performance tuning, backup/recovery, and high-availability systems. Proficient in tools \nlike Apica (External, Internal, Dynamic), GBT Dashboard, Grafana, ServiceNow, Java and Networking \nSupport, BigPanda, Jira and PagerDuty. \n \nWORK EXPERIEN

### Embeddings

In [69]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity




In [70]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """Initializes the embedding manager with the specified model.
        
        Args:
            model_name (str): The name of the SentenceTransformer model to use for embedding generation.
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Loads the SentenceTransformer model."""
        try:
            self.model = SentenceTransformer(self.model_name)
            print(f"Loaded embedding model: {self.model_name}")
            print(f"Model dimension: {self.model.get_embedding_dimension()} dimensions")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts.
        
        Args:
            texts (List[str]): A list of text strings to generate embeddings for.
        
        Returns:
            np.ndarray: An array of embedding vectors for the input texts.
        """
        if not self.model:
            raise ValueError("Embedding model is not loaded.")

        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings=self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings
    

### Initialize the embedding manager
embedding_manager=EmbeddingManager()
embedding_manager


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4042.80it/s]


Loaded embedding model: all-MiniLM-L6-v2
Model dimension: 384 dimensions


### Vector Store

In [98]:
class VectorStore:
    """Manages a vector store using ChromaDB for efficient similarity search."""

    def __init__(self, collection_name: str = "documents", persist_directory: str = "./chroma_db"):
        """Initializes the vector store with the specified collection name and persistence directory.
        
        Args:
            collection_name (str): The name of the ChromaDB collection to use for storing vectors.
            persist_directory (str): The directory where the ChromaDB database will be persisted.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_chromadb()
    
    def _initialize_chromadb(self):
        """Initializes the ChromaDB client and collection."""
        import os
        try:

            ### Initialize chromadb client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            ### Get or create the collection for storing document vectors
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "Collection for storing document embeddings"}
            )
            print(f"Vector store initialized: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")
        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    
    def add_documents(self, documents: List[Any], metadatas: List[dict], embeddings: np.ndarray):
        if not self.collection:
            raise ValueError("Vector store collection is not initialized.")

        try:
            ids = [str(uuid.uuid4()) for _ in range(len(documents))]

            self.collection.add(
                ids=ids,
                documents=documents,          # actual text
                metadatas=metadatas,          # metadata
                embeddings=embeddings.tolist()
            )

            print(f"Added {len(documents)} documents to the vector store.")
            print(f"Total documents in collection after addition: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vector_store=VectorStore()
vector_store
    

Vector store initialized: documents
Existing documents in collection: 144


In [101]:
### Convert Text to Embeddings
texts = [doc.page_content for doc in chunks]

### Generate embeddings for the document chunks
embeddings = embedding_manager.generate_embeddings(texts)

### Store the chunks and their embeddings in the vector store
vector_store.add_documents(
    documents=[doc.page_content for doc in chunks],
    metadatas=[doc.metadata for doc in chunks],
    embeddings=embeddings
)
vector_store.collection.count()

Generating embeddings for 18 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.07it/s]

Generated embeddings with shape: (18, 384)
Added 18 documents to the vector store.
Total documents in collection after addition: 162


162

### Retriever Pipeline from VectorStore

In [102]:
class RAGRetriever:
    """Implements a Retrieval-Augmented Generation (RAG) retriever using ChromaDB."""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """Initializes the RAG retriever with the specified vector store and embedding manager.
        
        Args:
            vector_store (VectorStore): An instance of the VectorStore class for managing document vectors.
            embedding_manager (EmbeddingManager): An instance of the EmbeddingManager class for generating embeddings.
        """
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """Retrieves the most relevant documents for a given query.
        
        Args:
            query (str): The input query string for which to retrieve relevant documents.
            top_k (int): The number of top relevant documents to retrieve.
            score_threshold (float): The minimum cosine similarity score required for a document to be considered relevant.  
        
        Returns:
            List[Dict[str, Any]]: A list of dictionaries containing retrieved documents with their metadata and relevance scores.
        """
        ### Generate embedding for the query
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        ### Search for relevant documents in the vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k,
            )
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    score = 1 - distance
                    if score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "score": score,
                            "distance": distance,
                            "rank": i + 1
                        })
                print(f"Retrieved {len(retrieved_docs)} relevant documents for the query: '{query}'")
            else:
                print(f"No relevant documents found for the query: '{query}'")
            
            return retrieved_docs
        except Exception as e:
            print(f"Error retrieving documents: {e}")
            raise


### Initialize the RAG retriever
rag_retriever = RAGRetriever(vector_store, embedding_manager)
rag_retriever

In [104]:
rag_retriever.retrieve("What is the introduction to LLMs?")

Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 83.32it/s]

Generated embeddings with shape: (1, 384)
Retrieved 0 relevant documents for the query: 'What is the introduction to LLMs?'


[]